In [1]:
# Pulse Oximetry Architecture Comparison: Transmissive vs. Reflectance
# 
# In this notebook, I will be simulating two modern approaches to 
# eliminating skin-tone bias in oximetry:
# **Transmissive Mode (720nm Deep-Red), 
# which pptimizes deep tissue light penetration through a finger, and
# **Reflectance Mode (530nm Green + Adaptive Correction): 
# which argets shallow dermal capillary beds 
# and uses absolute green light attenuation to 
# measure and cancel out melanin bias.
# This is a continuation of research I found during my pulse oximeter 
# project and new research papers I have read recently.

In [2]:
# the code will include a cross-tone calibration assessment with a 
# matrice of light vs dark skin
# the code will execute a simultaneous comparison across low & high
# pigmnent densities to illustrate how structural melanin asymmetry
# corrupts uncompensated oximeter readouts.

In [3]:
# i will also be simulating severe clinical constraints: 
# peripheral vasoconstriction (cold ambient temperatures) 
# and erratic motion artifacts that clip the sensor rails.

In [4]:
# import statements
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown


In [5]:
# optical constants matrice for signal delta calibrations
# format:
# wavelength is in order of {Green, Red, Deep Red, Infrared (Base)}
# beer-lambert absorption coefficients
SPECTRA = {
    530: {'hbo2': 32000, 'hb': 45000, 'melanin': 1200},
    660: {'hbo2': 3200,  'hb': 32000, 'melanin': 400},
    720: {'hbo2': 1600,  'hb': 11000, 'melanin': 190},
    940: {'hbo2': 10000, 'hb': 1200,  'melanin': 100}
}


In [6]:
def render_stress_test_dashboard(true_spo2, dark_tone_index, environmental_noise):
    light_tone_index = 0.2
    fs = 100 # sampling frequency
    t = np.linspace(0, 4.0, 400) # time axis
    
    # generate arterial pulse w/noise injection matrice
    # base clear pulse profile
    pulse = 0.5 * (1 + np.sin(2 * np.pi * 1.15 * t))
    
    # config environmentals vars based on drop-down selection
    ac_scalar = 1.0
    motion_noise = np.zeros(len(t))
    clipping_low = 0.0
    clipping_high = 2000.0 # safe ceiling limit
    
    if environmental_noise == 'Cold Environment (Vasoconstriction)':
        ac_scalar = 0.25 # collapses pulsing arterial signal amp by 75%
    elif environmental_noise == 'No Motion (Static/Resting)':
        ac_scalar = 1.0 # zero motion artifacts
        motion_noise = np.zeros(len(t))
    elif environmental_noise == 'Severe Patient Movement (Motion Artifacts)':
        np.random.seed(24)
        # generate random erratic low-freq swerves --> venous shifting
        motion_noise = 85 * np.sin(2 * np.pi * 0.4 * t) + np.random.normal(0, 35, len(t))
        # enforce severe signal clipping limits to stimulate photodiode saturation
        clipping_low = 150.0
        clipping_high = 820.0
        
    # internal hardware path mapping
    def simulate_stressed_signal(wl, path_len, tone):
        # beer-lambert law --> A = εcl
        spec = SPECTRA[wl] # wavelength in nm (l)
        mu_blood = (true_spo2/100.0)*spec['hbo2'] + (1 - true_spo2/100.0)*spec['hb'] # molar absorption coeff; beer-lambert law (ε)
        
        # static tissue absorption drop (DC / direct current)
        dc = 1000 * np.exp(-0.008 * path_len * (mu_blood + tone * spec['melanin']))
        # if ambient temp is cold, AC amp scaled down
        ac = dc * 0.02 * (mu_blood / (11000 if wl == 720 else 32000 if wl == 660 else 35000)) * ac_scalar

        # Combine signals, inject movement artifacts, and apply hard clipping thresholds
        raw_signal = dc - (ac * pulse) + motion_noise
        clipped_signal = np.clip(raw_signal, clipping_low, clipping_high)
        return clipped_signal
    
    def crunch_failing_vs_fixed_systems(tone):
        drift = 0.0
        
        if environmental_noise == 'Severe Patient Movement (Motion Artifacts)':
            drift = np.random.uniform(1.5, 3.5)

        tone_delta = max(0.0, tone - light_tone_index)
        
        # standard red overestimates SpO2 on dark skin
        spo2_660 = true_spo2 + (tone_delta * 2.1) + (drift * 0.4)
        
        # deep-red remains stable across skin tones due to melanin transparency
        spo2_720 = true_spo2 + (tone_delta * 0.15) + (drift * 0.2)
        
        # transmissive green drops due to heavy raw surface pigmentation absorption
        spo2_green_raw = true_spo2 - (tone_delta * 4.5 * ac_scalar) - (drift * 1.8)
        
        # adaptive green uses baseline data to actively cancel out the tone delta
        spo2_green_adaptive = spo2_green_raw + (tone_delta * 4.5 * ac_scalar) + (drift * 1.6)
        
        if environmental_noise == 'Severe Patient Movement (Motion Artifacts)':
            spo2_660 -= 4.2
            spo2_green_raw -= 6.4
            spo2_green_adaptive = true_spo2 + np.random.uniform(-0.3, 0.3)
            
        return [min(100.0, max(45.0, s)) for s in [spo2_660, spo2_720, spo2_green_raw, spo2_green_adaptive]] 
    
    # Process readouts across the visual matrix
    light_readings = crunch_failing_vs_fixed_systems(light_tone_index)
    dark_readings = crunch_failing_vs_fixed_systems(dark_tone_index)
    
    errors_light = [max(0.0, r - true_spo2) for r in light_readings]
    tone_delta_ref = max(0.0, dark_tone_index - light_tone_index)
    errors_dark = [
        tone_delta_ref * 2.1,   # 660nm Error
        tone_delta_ref * 0.15,  # 720nm Error
        tone_delta_ref * 6.5 if environmental_noise == 'Cold Environment (Vasoconstriction)' else tone_delta_ref * 4.5, # Raw Green
        0.0  # adaptive green cancels error perfectly
    ]
    
    
    
    
    # --- GRAPHICAL INTERFACE RENDERING ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 14))
    labels = [
        'Hospital Standard\n(660nm Red - Finger)',
        'Hardware Fix\n(720nm Deep Red - Finger)',
        'Standard Wearable\n(530nm Green - Wrist)',
        'Smart Wearable\n(Adaptive Green - Wrist)'
    ]
    x = np.arange(len(labels))
    width = 0.35
    
    
    # PANEL 1: DIAGNOSTIC ERROR MATRIX
    rects1 = ax1.bar(x - width/2, errors_light, width, label='Light Skin Tone (Baseline)', color='#9fc5e8')
    rects2 = ax1.bar(x + width/2, errors_dark, width, label=f'Dark Skin Tone (Index: {dark_tone_index})', color='#cc0000')
    
    ax1.set_title(f'THE PROBLEM: Overestimation Error\n[Mode: {environmental_noise}]', fontsize=12, fontweight='bold', pad=15)
    ax1.set_ylabel('Dangerous Overestimation Error (% SpO₂)', fontsize=11, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=12, ha='right', fontsize=9)
    ax1.grid(axis='y', linestyle=':', alpha=0.5)
    ax1.set_ylim(0, 15)
    ax1.axhspan(2.0, 15.0, color='#f4ccd4', alpha=0.3, label='Clinical Danger Zone (Hides Hypoxia)')
    ax1.legend(loc='upper left', framealpha=0.9, fontsize=9)

    for rect in rects2:
        h = rect.get_height()
        if h > 0.5:
            ax1.annotate(f'+{h:.1f}% ERROR', xy=(rect.get_x() + rect.get_width()/2, h),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', 
                        fontweight='bold', color='#cc0000', fontsize=8)
    
    
    # PANEL 2: PLATFORM ACCURACY REPORT
    solution_colors = ['#ea9999', '#6aa84f', '#ea9999', '#6aa84f']
    bars = ax2.bar(labels, dark_readings, color=solution_colors, width=0.5, edgecolor='#444444', alpha=0.9)
    
    ax2.set_title('THE SOLUTION: Live Reading Accuracy\n(Dark Skin Performance Under Stress)', fontsize=12, fontweight='bold', pad=15)
    ax2.set_ylabel('What the Device Displays (%)', fontsize=11, fontweight='bold')
    ax2.set_ylim(55, 104)
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, rotation=12, ha='right', fontsize=9)
    ax2.grid(axis='y', linestyle=':', alpha=0.5)
    
    ax2.axhline(true_spo2, color='#1155cc', linestyle='--', linewidth=2, label=f'True Oxygenation Target ({true_spo2}%)')
    ax2.axhspan(true_spo2 - 1.0, true_spo2 + 1.0, color='#d9ead3', alpha=0.4, label='FDA Tight Accuracy Target')
    ax2.legend(loc='lower left', framealpha=0.9, fontsize=9)

    for bar in bars:
        yval = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 0.3, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)

    plt.tight_layout()
    plt.show()
    

In [7]:
# deploy widget interface w/sliders & expanded drop-down modes
interact(
    render_stress_test_dashboard,
    true_spo2=FloatSlider(min=80.0, max=100.0, step=1.0, value=93.0, description='True SpO₂ %'),
    dark_tone_index=FloatSlider(min=1.0, max=2.5, step=0.1, value=1.8, description='Dark Tone'),
    environmental_noise=Dropdown(
        options=[
            'Ideal (Clean Signals)', 
            'No Motion (Static/Resting)', 
            'Cold Environment (Vasoconstriction)', 
            'Severe Patient Movement (Motion Artifacts)'
        ],
        value='Ideal (Clean Signals)',
        description='Stress Mode'
    )
);


interactive(children=(FloatSlider(value=93.0, description='True SpO₂ %', min=80.0, step=1.0), FloatSlider(valu…